# Fill Form — Gipformer + Silero VAD
Project độc lập: thu âm → chốt đoạn theo khoảng nghỉ → nhận dạng → điền trường đã chọn.

1. Đẩy thư mục `fill-form` lên repo ASR hiện tại, hoặc đẩy nội dung bên trong vào một repo riêng.
2. Chọn runtime **T4 GPU**, đặt `REPO_URL`/`REPO_BRANCH` ở cell đầu.
3. Chạy đến cell xem URL, mở link và cấp quyền microphone. Không cần đăng nhập hay upload ZIP.
4. Chỉ chạy cell dừng server khi đã sử dụng xong.

Giá trị form chỉ ở RAM của phiên; tải JSON để giữ kết quả. File audio upload được Gradio lưu tạm trên server. Ai có link public đều có thể dùng ứng dụng.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, time, re, signal

FAKE = False  # True: giao diện + transcript giả, không tải model.
SHARE = True
REPO_URL = 'https://github.com/Shinoaki0145/ASR.git'
REPO_BRANCH = ''  # Để trống để dùng nhánh mặc định
REPO_DIR = Path('/content/ASR') if Path('/content').exists() else Path.cwd()

def find_project(root):
    for folder in (root/'fill-form', root):
        if (folder/'app.py').is_file() and (folder/'fill_form/backend.py').is_file():
            return folder.resolve()

PROJECT = next((found for root in (REPO_DIR, Path.cwd(), Path.cwd().parent)
                if (found := find_project(root)) is not None), None)
if PROJECT is None:
    if not REPO_DIR.exists():
        if not REPO_URL: raise RuntimeError('Điền REPO_URL của repo chứa fill-form.')
        command = ['git', 'clone'] + (['--branch', REPO_BRANCH] if REPO_BRANCH else []) + [REPO_URL, str(REPO_DIR)]
        subprocess.run(command, check=True)
    PROJECT = find_project(REPO_DIR)
if PROJECT is None:
    raise RuntimeError('Chưa tìm thấy fill-form trong checkout. Đẩy code mới lên GitHub rồi cập nhật checkout hoặc dùng runtime mới. Notebook không tự ghi đè source.')
print('Project:', PROJECT)


In [ ]:
try:
    from dotenv import dotenv_values
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv>=1,<2'], check=True)
    from dotenv import dotenv_values
candidates = [PROJECT/'.env', Path('/content/.env')]
ENV_FILE = Path(os.environ['ASR_ENV_PATH']) if os.environ.get('ASR_ENV_PATH') else next((p for p in candidates if p.is_file()), candidates[0])
SERVER_ENV = {k:v for k,v in dotenv_values(ENV_FILE).items() if v is not None} if ENV_FILE.is_file() else {}
SERVER_ENV.update(os.environ)
SERVER_ENV.update(ASR_ENV_PATH=str(ENV_FILE), WEB_FAKE=str(FAKE).lower(), WEB_SHARE=str(SHARE).lower(), WEB_HOST='0.0.0.0' if SHARE else '127.0.0.1')
SERVER_ENV.setdefault('SEGMENT_SILENCE_SECONDS', '0.7')
print('Đã cấu hình; không cần LLM key hoặc đăng nhập web.')


In [ ]:
if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
EXTRAS = [] if FAKE else ['--extra', 'cuda']
if not FAKE:
    if not shutil.which('nvidia-smi'): raise RuntimeError('Chọn runtime T4 GPU trước')
    subprocess.run(['nvidia-smi'], check=True)
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([UV, 'sync', '--locked', *EXTRAS], cwd=PROJECT, check=True)
if not FAKE:
    subprocess.run([UV, 'run', '--locked', *EXTRAS, 'python', '-m', 'fill_form.setup', '--config', str(PROJECT/'config.toml')], cwd=PROJECT, env=SERVER_ENV, check=True)
print('Môi trường sẵn sàng.')


In [ ]:
def stop_server():
    process = globals().get('FORM_PROCESS')
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=12)
        except subprocess.TimeoutExpired:
            process.terminate()
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=5)
    log = globals().get('FORM_LOG')
    if log is not None: log.close()
stop_server()
LOG_PATH = PROJECT/'form-server.log'
FORM_LOG = LOG_PATH.open('w')
FORM_PROCESS = subprocess.Popen([UV, 'run', '--locked', *EXTRAS, 'python', '-u', 'app.py'], cwd=PROJECT, env=SERVER_ENV, stdout=FORM_LOG, stderr=subprocess.STDOUT)
print('Server PID:', FORM_PROCESS.pid)


In [ ]:
from IPython.display import display, Markdown
for _ in range(25):
    time.sleep(2)
    content = LOG_PATH.read_text(errors='replace')
    urls = re.findall(r'https://[^\s]+\.gradio\.live', content)
    if urls or FORM_PROCESS.poll() is not None: break
print(content[-6000:])
if urls: display(Markdown(f'[Mở ASR điền form]({urls[-1]})'))
print('Server:', 'đang chạy' if FORM_PROCESS.poll() is None else f'đã dừng, exit={FORM_PROCESS.returncode}')


Chọn trường, bật mic và nói. Nghỉ khoảng 0,7 giây để chốt đoạn. Các đoạn tiếp theo nối vào cùng trường. Để sửa/xóa, nhập toàn bộ giá trị ở ô sửa và bấm **Lưu sửa tay**. Tắt mic, chờ xử lý xong và kiểm tra các trường trước khi **Tải JSON**.

Chạy cell dưới khi đã sử dụng xong (dữ liệu chưa tải sẽ mất).

Hoặc tắt mic, chọn trường rồi bấm **Upload file âm thanh** (tối đa 50 MB / 5 phút). File đi qua VAD theo thời gian thực và điền vào trường đã chọn lúc upload. Chờ xử lý xong trước khi tải JSON.


In [ ]:
stop_server()
print('Đã dừng server.')
